# Fix resource files

In [22]:
import pandas as pd

In [27]:
def remove_retirements(str): 
    df = pd.read_csv(str) 
    df["Can_Retire"] = 0
    df["New_Build"] = 0 
    df["capacity_factor"] = 1.0
    df["minimum_load_mw"] = df[["minimum_load_mw", "Cap_Size"]].min(axis=1)
    if "gen_forced_outage_rate" in df.columns:
        df["gen_forced_outage_rate"] = 0.0
    if "gen_scheduled_outage_rate" in df.columns:
        df["gen_scheduled_outage_rate"] = 0.0
    
    df.to_csv(str, index=False)

In [28]:
for filename in ["Hydro.csv", "Must_run.csv", "Storage.csv", "Thermal.csv", "Vre.csv"]:
    remove_retirements(filename)

In [30]:
# 1. Load the original CSV files
thermal_df = pd.read_csv('Thermal.csv')
must_run_df = pd.read_csv('Must_run.csv')

# 2. Identify the nuclear rows in Thermal.csv 
# (Checking both 'Resource' and 'gen_type' just to be safe)
is_nuclear = thermal_df['Resource'].str.contains('nuclear', case=False, na=False) | \
             thermal_df['gen_type'].str.contains('Nuclear', case=False, na=False)

# 3. Apply the flexible baseload fixes
# Set Start Cost to $0 so the solver doesn't penalize turning them on
thermal_df.loc[is_nuclear, 'Start_Cost_per_MW'] = 0.0

# Set Min Power to 0.95 so they run as baseload but leave 5% headroom for reserves
thermal_df.loc[is_nuclear, 'Min_Power'] = 0.50

# 4. Save the updated dataset
thermal_df.to_csv('Thermal.csv', index=False)

print(f"Successfully updated {is_nuclear.sum()} nuclear units in Thermal.csv!")

Successfully updated 8 nuclear units in Thermal.csv!


In [31]:
hydro_df = pd.read_csv('Hydro.csv')
# 2. Remove the columns that trigger the "Reservoir Storage" math
columns_to_drop = ['Hydro_Energy_to_Power_Ratio', 'LDS', 'variable_CF']

# Using errors='ignore' ensures the script won't crash if you already deleted one
hydro_df = hydro_df.drop(columns=columns_to_drop, errors='ignore')

# 3. Zero out the minimum load to prevent infeasibility during severe droughts
if 'minimum_load_mw' in hydro_df.columns:
    hydro_df['minimum_load_mw'] = 0.0

# 4. Save the fixed data back to the CSV (overwriting the old one)
hydro_df.to_csv('Hydro.csv', index=False)

print("Hydro.csv successfully converted to run-of-river!")


Hydro.csv successfully converted to run-of-river!


In [32]:
import pandas as pd

# 1. Load the original files
hydro_df = pd.read_csv('Hydro.csv')
vre_df = pd.read_csv('Vre.csv')

# 2. Strip out all the thermal/reservoir columns from the Hydro plants
cols_to_drop = ['Hydro_Energy_to_Power_Ratio', 'LDS', 'Eff_Down', 'Eff_Up', 
                'Ramp_Dn_Percentage', 'Ramp_Up_Percentage', 'Min_Power', 'variable_CF']

hydro_df = hydro_df.drop(columns=[c for c in cols_to_drop if c in hydro_df.columns])

# 3. Ensure minimum load is 0 so the solver never gets trapped during a drought
if 'minimum_load_mw' in hydro_df.columns:
    hydro_df['minimum_load_mw'] = 0.0

# 4. Add the required VRE columns to the Hydro dataframe so they match perfectly
hydro_df['Num_VRE_Bins'] = 1
hydro_df['Max_Cap_MW'] = 0.0
hydro_df['Inv_Cost_per_MWyr'] = 0.0

# 5. Concatenate them together (append hydro to the bottom of VRE)
combined_vre = pd.concat([vre_df, hydro_df], ignore_index=True)

# 6. Fill any leftover blanks (like VRE's "cost_case" column) with 0 or empty strings
combined_vre = combined_vre.fillna(0)

# 7. Save the new, fully merged VRE file
combined_vre.to_csv('VRE_Combined.csv', index=False)

# 8. Empty out the old Hydro.csv file but keep the headers so GenX doesn't crash looking for it
empty_hydro = hydro_df.iloc[0:0]
empty_hydro.to_csv('Hydro.csv', index=False)

print("Hydro plants successfully prepared and appended to VRE_Combined.csv!")

Hydro plants successfully prepared and appended to VRE_Combined.csv!
